In [1]:
import os
import warnings
from pathlib import Path
from datetime import datetime
from typing import Any, Optional
import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from tqdm import tqdm
import ipywidgets as widgets
from ipyleaflet import WidgetControl
from matplotlib.lines import Line2D
warnings.filterwarnings('ignore')


PROJECT_NAME = 'optical-depth-468907-t1'
SHP_PATH = 'parcelas 2025.shp'
START_DATE = '2025-04-01'
END_DATE = '2025-08-15'
CLOUD_THRESHOLD = 10
ID_COLUMN = 'id'

OUTPUT_DIR = "C:/Users/kin/Desktop/Congreso/graficas_parcelas"
CSV_DIR = 'C:/Users/kin/Desktop/Congreso/csv'

EVENTS_CSV = 'C:/Users/kin/Desktop/Congreso/events.csv'
NAME_COLUMN = "Nombre" 

TMP_GEOJSON = os.path.join(OUTPUT_DIR, 'indices_fc.geojson')
INDEX_LIST = ['NDVI','SAVI','EVI','GNDVI','NDWI']

try:
    ee.Initialize(project=PROJECT_NAME)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_NAME)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)


def load_shapefile(path: str, id_col: str) -> gpd.GeoDataFrame:
    """
    Carga un archivo shapefile en un GeoDataFrame y valida la existencia de una columna identificadora.

    Args:
        path (str): Ruta al archivo shapefile
        id_col (str): Nombre de la columna que actuará como identificador único de cada geometría.

    Returns:
        gpd.GeoDataFrame
    """

    # Leer el shapefile y convertir a WGS 84
    gdf = gpd.read_file(path).to_crs(4326)

    # Filtrar para mantener solo geometrías válidas, no vacías y no nulas
    gdf = gdf[
        gdf.geometry.notna() &              # Eliminar filas con geometría nula
        ~gdf.geometry.is_empty &            # Eliminar geometrías vacías
        gdf.geometry.is_valid               # Mantener solo geometrías válidas topológicamente
    ].copy()

    if id_col not in gdf.columns:
        raise ValueError(f"Column '{id_col}' not found in shapefile.")

    return gdf


def export_fc_to_geojson(fc: ee.FeatureCollection, out_path: str) -> None:
    """
    Exporta una FeatureCollection a un archivo GeoJSON.

    Args:
        fc (ee.FeatureCollection): Objeto FeatureCollection de Earth Engine que contiene las entidades geográficas a exportar
        out_path (str): Ruta completa del archivo de salida

    """

    # Verificar si ya existe un archivo en la ruta de salida
    if os.path.exists(out_path):
        os.remove(out_path)  # Eliminar el archivo previo para garantizar que la exportación sea limpia

    geemap.ee_export_vector(fc, filename=out_path)

def mask_clouds(img: ee.Image) -> ee.Image:
    """
    Aplica una máscara a una imagen Sentinel-2 para eliminar píxeles cubiertos por nubes y sus sombras.

    Args:
        img (ee.Image): Imagen de entrada de Sentinel-2 con bandas de calidad (QA60) y clasificación de escena (SCL).

    Returns:
        ee.Image: Imagen con los píxeles nublados o con sombra enmascarados (marcados como no válidos).
    """
    # Extraer la banda QA60 (máscara de calidad) que contiene información de nubes y sombras
    qa = img.select('QA60')

    # Desplazar los bits 10 y 11 a la derecha para aislar la información de nubes
    # bitwiseAnd(3) = 00000011 en binario, para extraer solo esos 2 bits
    # gt(0) = 1 si hay nubes detectadas en esos bits
    cloud_qa = qa.rightShift(10).bitwiseAnd(3).gt(0)

    # Extraer la banda SCL (Scene Classification Layer) que clasifica cada píxel en diferentes categorías
    scl = img.select('SCL')

    # Detectar nubes y sombras de nubes a partir de la clasificación:
    # eq(3)  = sombreado de nubes
    # gte(8) & lte(10) = clases 8, 9 y 10 (nubes medias, densas y cirros)
    cloud_scl = scl.eq(3).Or(scl.gte(8).And(scl.lte(10)))

    # Combinar ambas máscaras (QA y SCL), invertirlas con Not() para obtener píxeles válidos
    mask = cloud_qa.Or(cloud_scl).Not()

    return img.updateMask(mask)


INDEX_VIZ = {
    'NDVI': dict(min=0.0, max=1.0, palette=['#d73027','#f46d43','#fdae61','#fee08b','#d9ef8b','#a6d96a','#66bd63','#1a9850']),
    'SAVI': dict(min=0.0, max=1.5, palette=['#d73027','#f46d43','#fdae61','#fee08b','#d9ef8b','#a6d96a','#66bd63','#1a9850']),
    'EVI': dict(min=0.0, max=1.0, palette=['#ffffcc','#c2e699','#78c679','#31a354','#006837']),
    'GNDVI': dict(min=0.0, max=1.0, palette=['#f7fcf5','#e5f5e0','#c7e9c0','#a1d99b','#74c476','#31a354','#006d2c']),
    'NDWI': dict(min=-1.0, max=1.0, palette=['#f7fbff','#deebf7','#c6dbef','#9ecae1','#6baed6','#3182bd','#08519c']),
}

def add_indices(img: ee.Image, indices: list) -> ee.Image:
    """
    Calcula y agrega índices espectrales seleccionados a una imagen Sentinel-2.
    """
    # Escalar los valores de reflectancia dividiendo entre 10000
    img_scaled = img.divide(10000)

    bands = {
        # NDVI: Índice de Vegetación de Diferencia Normalizada
        'NDVI': lambda x: x.normalizedDifference(['B8', 'B4']),
        
        # SAVI: Índice de Vegetación Ajustado al Suelo (factor de corrección L=0.5)
        'SAVI': lambda x: x.expression(
            '(NIR - RED) / (NIR + RED + 0.5) * 1.5',
            {'NIR': x.select('B8'), 'RED': x.select('B4')}
        ),
        
        # EVI: Índice de Vegetación Mejorado
        'EVI': lambda x: x.expression(
            '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
            {'NIR': x.select('B8'), 'RED': x.select('B4'), 'BLUE': x.select('B2')}
        ),
        
        # GNDVI: Índice de Vegetación de Diferencia Normalizada Verde
        'GNDVI': lambda x: x.normalizedDifference(['B8', 'B3']),
        
        # NDWI: Índice de Agua de Diferencia Normalizada
        'NDWI': lambda x: x.normalizedDifference(['B8', 'B11']),
        # IAF (LAI) estimado a partir de NDVI
        # Fórmula: LAI = -( ln( (0.69 - NDVI) / 0.59 ) ) / 0.91
        'IAF': lambda x: x.expression(
            '-(log((0.69 - NDVI) / 0.59)) / 0.91',
            {'NDVI': x.normalizedDifference(['B8', 'B4'])}
        ),
    }

    for idx in indices:
        if idx in bands:
            # Calcular el índice usando la fórmula definida en 'bands' y añadirlo como nueva banda
            img_scaled = img_scaled.addBands(
                bands[idx](img_scaled).rename(idx)
            )

    # Retornar la imagen resultante, manteniendo las propiedades originales (metadatos) de la imagen de entrada
    return img_scaled.copyProperties(img, img.propertyNames())

def preprocess_image(img: ee.Image, indices: list) -> ee.Image:
    return add_indices(mask_clouds(img), indices)


def extract_stats(img, regions, indices: list) -> ee.FeatureCollection:
    """
    Calcula estadísticas descriptivas de índices espectrales para regiones específicas
    a partir de una imagen de Google Earth Engine.

    Args:
        img (ee.Image | str): Imagen de entrada o ID de imagen de Earth Engine 
                              que contiene las bandas/índices a analizar.
        regions (ee.FeatureCollection | str): Colección de polígonos o regiones (o su ID) 
                                               sobre las que se calcularán las estadísticas.
        indices (list): Lista de nombres de las bandas/índices a incluir en el análisis.

    Returns:
        ee.FeatureCollection: FeatureCollection con las estadísticas calculadas para cada región.
    """
    # Asegurarse de que 'img' es un objeto ee.Image
    img = ee.Image(img)

    # Asegurarse de que 'regions' es un objeto ee.FeatureCollection
    regions = ee.FeatureCollection(regions)

    # Definir un reductor combinado con varias métricas:
    reducer = (
        ee.Reducer.minMax()                              # Valor mínimo y máximo
        .combine(ee.Reducer.mean(), sharedInputs=True)   # Media
        .combine(ee.Reducer.stdDev(), sharedInputs=True) # Desviación estándar
        .combine(ee.Reducer.count(), sharedInputs=True)  # Conteo de píxeles válidos
        .combine(                                       # Percentiles 0, 25, 50, 75 y 100
            ee.Reducer.percentile([0, 25, 50, 75, 100]),
            sharedInputs=True
        )
    )

    # Calcular las estadísticas para cada región en 'regions'
    stats = img.select(indices).reduceRegions(
        collection=regions,  # Regiones sobre las que se calcula
        reducer=reducer,     # Estadísticas a obtener
        scale=10             # Escala en metros por píxel (Sentinel-2 = 10 m)
    )

    date = ee.Date(img.get('system:time_start'))

    # Añadir la fecha y el día del año (DOY) como atributos a cada región
    return stats.map(lambda f: ee.Feature(f).set({
        'date': date.format('YYYY-MM-dd'),       # Fecha formateada
        'doy': date.getRelative('day', 'year')  # Día relativo dentro del año
    }))


def process_image(im, regions, indices):
    im = ee.Image(im)
    return extract_stats(preprocess_image(im, indices), regions, indices)

def build_fc(regions: ee.FeatureCollection, start: str, end: str, cloud_th: int, indices: list) -> ee.FeatureCollection:
    """
    Construye una FeatureCollection con estadísticas de índices espectrales para un conjunto de regiones
    usando imágenes Sentinel-2 filtradas por fecha y porcentaje de nubosidad.

    Args:
        regions (ee.FeatureCollection): Colección de regiones (polígonos) sobre las que se calcularán las estadísticas.
        start (str): Fecha inicial del rango de análisis en formato 'YYYY-MM-DD'.
        end (str): Fecha final del rango de análisis en formato 'YYYY-MM-DD'.
        cloud_th (int): Umbral máximo de porcentaje de nubes permitido para las imágenes (0–100).
        indices (list): Lista de índices espectrales que se calcularán en cada imagen.

    Returns:
        ee.FeatureCollection: FeatureCollection combinada con las estadísticas de todas las imágenes que cumplen
                              los filtros y para todas las regiones especificadas.
    """
    regions = ee.FeatureCollection(regions)

    # Cargar la colección de imágenes Sentinel-2 SR
    s2 = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(regions)  # Mantener solo imágenes que cubran las regiones de interés
        .filterDate(start, end) # Filtrar por rango de fechas
        .filter(                # Filtrar por porcentaje de nubes en metadatos
            ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_th)
        )
    )
    # Incluye pasos como enmascarar nubes, calcular índices y extraer estadísticas
    # Luego combinar todas las FeatureCollections resultantes en una sola
    return ee.FeatureCollection(
        s2.map(lambda im: process_image(im, regions, indices))
    ).flatten()

# ================================================================================================

def read_events_csv(path: str, id_col: str, name_col: str = "Nombre") -> Optional[pd.DataFrame]:
    """
    Lee el CSV de eventos (id, Nombre, y columnas de fechas) si existe.
    Convierte a datetime todas las columnas distintas de id y Nombre,
    asumiendo formato DD/MM/YYYY 
    """
    if not path or not os.path.exists(path):
        return None
    df = pd.read_csv(path)

    if id_col not in df.columns or name_col not in df.columns:
        return None

    # columnas candidatas a fechas = todas menos id y nombre
    date_cols = [c for c in df.columns if c not in [id_col, name_col]]

    # Parseo estricto a DD/MM/YYYY; si no coincide -> NaT
    for c in date_cols:
        df[c] = pd.to_datetime(df[c], format="%d/%m/%Y", errors="coerce")

    return df[[id_col, name_col] + date_cols]


def build_events_maps(events_df: pd.DataFrame, id_col: str, name_col: str):
    """
    Devuelve:
      - name_by_id: dict id -> Nombre
      - events_by_id: dict id -> dict { etiqueta_evento : [fechas válidas] }
    """
    if events_df is None or events_df.empty:
        return {}, {}

    date_cols = [c for c in events_df.columns if c not in [id_col, name_col]]
    name_by_id = dict(zip(events_df[id_col], events_df[name_col]))

    events_by_id = {}
    for _, row in events_df.iterrows():
        pid = row[id_col]
        evts = {}
        for c in date_cols:
            val = row[c]
            if pd.notna(val):
                evts.setdefault(c, []).append(pd.to_datetime(val))
        if evts:
            events_by_id[pid] = evts
    return name_by_id, events_by_id


def draw_event_vlines(ax, events: dict):
    """
    Dibuja líneas verticales por cada evento {label: [fechas]} y añade leyenda.
    """
    if not events:
        return

    handles, labels = ax.get_legend_handles_labels()
    new_handles, new_labels = [], []

    for evt_label, dates in events.items():
        if not isinstance(dates, (list, tuple, pd.Series)):
            dates = [dates]
        for d in dates:
            if pd.isna(d):
                continue
            ax.axvline(pd.to_datetime(d), linestyle="--", linewidth=1, alpha=0.8)
        new_handles.append(Line2D([0], [0], linestyle="--", linewidth=1))
        new_labels.append(evt_label)

    if new_handles:
        ax.legend(handles + new_handles, labels + new_labels, frameon=False, loc="best")


def plot_boxplot(df, parcel_id, index, output_dir, parcel_label=None, events=None):
    cols = [f"{index}_min", f"{index}_p25", f"{index}_p50", f"{index}_p75", f"{index}_p100"]
    if not all(c in df.columns for c in cols):
        return None

    df = df.dropna(subset=[f"{index}_p50"]).sort_values("date")
    if df.empty:
        return None

    box_data, dates = [], []
    for _, row in df.iterrows():
        try:
            box_data.append({
                'label': row['date'].strftime("%Y-%m-%d"),
                'whislo': float(np.ravel(row[cols[0]])[0]),
                'q1': float(np.ravel(row[cols[1]])[0]),
                'med': float(np.ravel(row[cols[2]])[0]),
                'q3': float(np.ravel(row[cols[3]])[0]),
                'whishi': float(np.ravel(row[cols[4]])[0]),
                'fliers': []
            })
            dates.append(row['date'])
        except:
            continue

    if not box_data:
        return None

    label_to_show = parcel_label if parcel_label else parcel_id

    fig, ax = plt.subplots(figsize=(12, 7), dpi=300)
    bp = ax.bxp(box_data, positions=mdates.date2num(dates), widths=6, showfliers=False, patch_artist=True)
    for box in bp['boxes']:
        box.set_facecolor("#4C72B0")
        box.set_alpha(0.6)

    ax.set_title(f"{index} - Boxplots\nParcela {label_to_show}")
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b-%Y"))
    fig.autofmt_xdate(rotation=30)
    ax.grid(True, linestyle="--", alpha=0.4)

    draw_event_vlines(ax, events)

    out_dir = Path(output_dir) / index / "boxplots"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"parcela_{parcel_id}.png"
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    return out_path


def plot_mean_line(df: pd.DataFrame, parcel_id: Any, index: str, output_dir: str, parcel_label: Optional[str] = None, events: Optional[dict] = None) -> Optional[Path]:
    mean_col = f"{index}_mean"
    if mean_col not in df.columns:
        return None

    df = df.dropna(subset=[mean_col]).sort_values("date")
    if df.empty:
        return None

    label_to_show = parcel_label if parcel_label else parcel_id

    fig, ax = plt.subplots(figsize=(12, 6), dpi=300)
    line_plot = ax.plot(
        df['date'], df[mean_col],
        marker="o", markersize=4,
        linewidth=1.5, color="#4C72B0",
        label=f"{index} (media)"
    )

    ax.set_title(f"{index} - Valores medios\nParcela {label_to_show}", pad=12)
    ax.set_ylabel(index)
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b-%Y"))
    fig.autofmt_xdate(rotation=30, ha="right")
    ax.grid(True, linestyle="--", alpha=0.4)

    ax.legend(frameon=False, loc="best")

    draw_event_vlines(ax, events)

    out_dir = Path(output_dir) / index / "mean_line"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"parcela_{parcel_id}.png"
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    return out_path


# ================== MAIN ===================

gdf = load_shapefile(SHP_PATH, ID_COLUMN)
parcelas_fc = geemap.geopandas_to_ee(gdf)

print("Downloading GEE data...")
indices_fc = build_fc(parcelas_fc, START_DATE, END_DATE, CLOUD_THRESHOLD, INDEX_LIST)
export_fc_to_geojson(indices_fc, TMP_GEOJSON)

features = gpd.read_file(TMP_GEOJSON)
features['date'] = pd.to_datetime(features['date'])

if len(INDEX_LIST) == 1:
    idx = INDEX_LIST[0]
    rename_map = {}
    col_map = {
        "min": f"{idx}_min",
        "max": f"{idx}_p100",
        "mean": f"{idx}_mean",
        "stdDev": f"{idx}_stdDev",
        "count": f"{idx}_count",
        "p0": f"{idx}_min",
        "p25": f"{idx}_p25",
        "p50": f"{idx}_p50",
        "p75": f"{idx}_p75",
        "p100": f"{idx}_p100"
    }
    for old, new in col_map.items():
        if old in features.columns and new not in features.columns:
            rename_map[old] = new
    if rename_map:
        features = features.rename(columns=rename_map)

if 'cultivo' not in features.columns:
    features['cultivo'] = 'N/A'

available_indices = [idx for idx in INDEX_LIST if f"{idx}_mean" in features.columns]

events_df = read_events_csv(EVENTS_CSV, ID_COLUMN, NAME_COLUMN)
name_by_id, events_by_id = build_events_maps(events_df, ID_COLUMN, NAME_COLUMN)

print("\nGenerating plots...")
for parcel_id in tqdm(features[ID_COLUMN].unique(), desc="Processing parcels"):
    df_parcel = features[features[ID_COLUMN] == parcel_id]
    parcel_label = name_by_id.get(parcel_id, None)
    parcel_events = events_by_id.get(parcel_id, None)

    for idx in available_indices:
        plot_boxplot(df_parcel, parcel_id, idx, OUTPUT_DIR, parcel_label=parcel_label, events=parcel_events)
        plot_mean_line(df_parcel, parcel_id, idx, OUTPUT_DIR, parcel_label=parcel_label, events=parcel_events)


for idx in available_indices:
    pivot = features.pivot_table(index=ID_COLUMN, columns='date', values=f"{idx}_mean")
    pivot.to_csv(os.path.join(CSV_DIR, f"{idx.lower()}_mean.csv"))

with open(os.path.join(OUTPUT_DIR, 'report.txt'), 'w') as f:
    f.write(f"Processing Report - {datetime.now()}\n")
    f.write(f"Parcels processed: {features[ID_COLUMN].nunique()}\n")
    f.write(f"Dates used: {features['date'].nunique()}\n")

print("\nProcess completed!")




ModuleNotFoundError: No module named 'geemap'

In [4]:
from ipywidgets import Dropdown, SelectionSlider, HBox, Layout
from ipyleaflet import FullScreenControl, WidgetControl

# ---
region = parcelas_fc.geometry()

base_ic = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(region)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_THRESHOLD))
    .map(lambda im: preprocess_image(ee.Image(im), INDEX_LIST))
    .map(lambda im: ee.Image(im).set({'date': ee.Date(im.get('system:time_start')).format('YYYY-MM-dd')}))
)

# --- Crear colecciones por índice ---
def create_index_collection(idx):
    return base_ic.map(lambda im: ee.Image(im).select(idx).rename('index').copyProperties(im, im.propertyNames()))

index_collections = {idx: create_index_collection(idx) for idx in INDEX_LIST}

# --- Fechas disponibles por índice ---
dates_by_index = {}
for idx, ic in index_collections.items():
    try:
        dates = ic.aggregate_array('date').distinct().sort().getInfo()
    except Exception:
        dates = []
    dates_by_index[idx] = dates



DEFAULT_VIZ = dict(min=0, max=1, palette=['white','black'])

def get_viz(idx):
    return INDEX_VIZ.get(idx, DEFAULT_VIZ)


valid_indices = [i for i in INDEX_LIST if dates_by_index.get(i)]
if not valid_indices:
    print("No hay fechas disponibles en el rango.")
    valid_indices = INDEX_LIST[:1]

idx_dd = Dropdown(
    options=valid_indices,
    value=valid_indices[0],
    description='Índice:',
    layout=Layout(width='220px')
)

date_dd = SelectionSlider(
    options=dates_by_index.get(idx_dd.value, []),
    value=dates_by_index.get(idx_dd.value, [None])[-1],
    description='Fecha:',
    continuous_update=False,
    layout=Layout(width='520px')
)

# --- Mapa base ---
try:
    center = gdf.unary_union.centroid
    center_lat, center_lon = center.y, center.x
except:
    bounds = gdf.total_bounds
    center_lat = (bounds[1] + bounds[3]) / 2
    center_lon = (bounds[0] + bounds[2]) / 2

m = geemap.Map(center=[center_lat, center_lon], zoom=15)
m.add_basemap('SATELLITE')
m.add_control(FullScreenControl())

# Parcelas
m.addLayer(parcelas_fc.style(**{
    'color': '#222222', 'width': 2, 'lineType': 'solid', 'fillColor': '00000000'
}), {}, 'Parcelas')

# --- Estado ---
current_layer = None
current_colorbar = None

def _remove_all_colorbars(the_map):
    for ctrl in list(the_map.controls):
        try:
            if isinstance(ctrl, WidgetControl):
                w = getattr(ctrl, "widget", None)
                if w and any(k in repr(w).lower() for k in ["branca", "colorbar", "colormap"]):
                    the_map.controls.remove(ctrl)
                    continue
            if any(k in repr(ctrl).lower() for k in ["colorbar", "branca", "colormap"]):
                the_map.controls.remove(ctrl)
        except:
            pass

def update_layer():
    global current_layer
    if current_layer:
        try:
            m.remove_layer(current_layer)
        except:
            pass
        current_layer = None

    idx = idx_dd.value
    date = date_dd.value
    if not idx or not date:
        return

    ic = index_collections[idx].filter(ee.Filter.eq('date', date))
    img = ic.median().clip(region)
    viz = get_viz(idx)
    current_layer = geemap.ee_tile_layer(img, viz, name=f'{idx} {date}', shown=True)
    m.add_layer(current_layer)

def on_index_change(change):
    global current_colorbar
    if change['name'] != 'value':
        return

    idx = change['new']
    available = dates_by_index.get(idx, [])
    date_dd.options = available
    date_dd.value = available[-1] if available else None

    _remove_all_colorbars(m)
    viz = get_viz(idx)
    current_colorbar = m.add_colorbar_branca(
        colors=viz['palette'],
        vmin=viz['min'],
        vmax=viz['max'],
        label=idx
    )

    update_layer()

# --- Observadores y render inicial ---
idx_dd.observe(on_index_change)
date_dd.observe(lambda change: update_layer(), names='value')

viz = get_viz(idx_dd.value)
current_colorbar = m.add_colorbar_branca(
    colors=viz['palette'],
    vmin=viz['min'],
    vmax=viz['max'],
    label=idx_dd.value
)
update_layer()

# --- Mostrar UI + Mapa ---
display(HBox([idx_dd, date_dd]))
display(m)


NameError: name 'parcelas_fc' is not defined

In [3]:
from ipyleaflet import FullScreenControl, WidgetControl
print("ipyleaflet OK")

ipyleaflet OK
